In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize

# ===============================
# 🔹 Data Loading
# ===============================

# (사용자 로컬 경로 → 상대 경로로 변경)
file_paths = [
    r"./data/data_0.txt",
    r"./data/data_1.txt",
    r"./data/data_2.txt",
    r"./data/data_3.txt"
]

colors = ['r', 'g', 'b', 'k']
labels = ['Sensor A', 'Sensor B', 'Sensor C', 'Sensor D']

processed_signals = []

for i, file_path in enumerate(file_paths):
    # Load data
    with open(file_path, 'r') as f:
        y_values = np.array([float(line.strip()) for line in f])

    x_values = np.arange(len(y_values))

    # Plot raw data
    plt.figure(figsize=(5, 2))
    plt.plot(x_values / 600, y_values, color=colors[i])
    plt.xlabel("Time [min]")
    plt.ylabel("nIR Intensity [V]")
    plt.title(f"Real-Time Monitoring [Line {i}]")
    plt.show()

    # Find onset point
    dy = np.diff(y_values)
    dy = np.concatenate([[0], dy])
    threshold = 0.3
    start_index = np.argmax(dy > threshold) + 100

    # Baseline
    baseline_window = 100
    baseline_end = start_index + baseline_window
    baseline = np.mean(y_values[start_index:baseline_end])

    # Normalize to baseline
    reaction_ratio = (y_values - baseline) / baseline * 100
    reaction_ratio = -reaction_ratio
    x_values = x_values - start_index

    # FFT smoothing
    reaction_fft = np.fft.fft(reaction_ratio)
    freqs = np.fft.fftfreq(len(reaction_ratio), d=1 / 600)
    cutoff_hz = 2
    reaction_fft[np.abs(freqs) > cutoff_hz] = 0
    smoothed = np.fft.ifft(reaction_fft).real

    # Interpolation
    x_time_min = x_values / 600
    interp_func = interp1d(x_time_min, smoothed, kind='cubic', fill_value="extrapolate")
    x_fit = np.linspace(0, 15, 3000)
    y_fit = interp_func(x_fit)

    # Offset correction
    min_val = np.min(y_fit)
    if min_val < 0:
        y_fit -= min_val
    y_fit /= 100

    processed_signals.append((x_fit, y_fit, labels[i], colors[i]))

# ===============================
# 🔹 Hill Model Fitting
# ===============================

def hill(x, A, B, C, N):
    x = np.clip(x, 1e-6, 1e6)
    return A * (x ** N) / (B ** N + x ** N) + C


# Parameter sets (익명 데이터셋)
param_sets = [
    {
        "A": [0.9, 0.8, 0.7, 0.9],
        "B": [0.001, 0.003, 1.0, 0.005],
        "C": [0.02, 0.07, 0.15, 0.07],
        "N": [0.9, 0.9, 0.5, 0.7]
    }
]

x_fit = np.linspace(0, 15, len(processed_signals[0][1]))
x0 = [0.001, 0.001, 0.01, 0.001]
x_all = [[] for _ in range(4)]

for t_idx, t in enumerate(x_fit):
    R_target = [sig[1][t_idx] for sig in processed_signals]

    def objective(x):
        x_sum = np.sum(x)
        error = 0
        for j in range(4):
            Ri_pred = 0
            for k in range(1):
                p = param_sets[k]
                Ri_pred += (x[k] / x_sum) * hill(x[k], p["A"][j], p["B"][j], p["C"][j], p["N"][j])
            error += (Ri_pred - R_target[j]) ** 2
        return error

    result = minimize(objective, x0=x0, bounds=[(0.0001, 0.1)] * 4)
    sol = result.x
    x0 = sol
    for i in range(4):
        x_all[i].append(sol[i])

# ===============================
# 🔹 Plot Results
# ===============================

fig, axs = plt.subplots(4, 1, figsize=(8, 8), sharex=True)
for i, (ax, color) in enumerate(zip(axs, colors)):
    ax.plot(x_fit, x_all[i], color=color)
    ax.set_ylabel(f"x{i+1}")
axs[-1].set_xlabel("Time [min]")
plt.tight_layout()
plt.show()
